In [ ]:
import sys
import pickle

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objects as go
import kaleido

In [ ]:
#REQUIRED FUNCTIONS

In [ ]:
def plot_data_single_panel_interactive(mpsat_results, initial_db, true_type, predicted_types, panel_label_symbol):
    """
    Plot a single panel for a specified polymer type using Plotly and save it as an interactive HTML file.
    """
    # Wavenumber positions for vertical lines
    wavenumbers = {
        'PE': [2915, 2845, 1472, 1467, 1377, 1462, 730, 717],
        'PA': [687, 1199, 1274, 1372, 1464, 1538, 1634, 2858, 2932, 3298],
        'PP': [2950, 2915, 2838, 1455, 1377, 1166, 997, 972, 840, 808],
        'PS': [3024, 2847, 1601, 1492, 1451, 1027, 694, 537],
        'PVC': [616, 966, 1099, 1255, 1331, 1427],
        'PU': [1223, 1451, 1531, 1731, 2865],
        'EVA': [720, 1020, 1241, 1469, 1740, 2848, 2917],
        'CPE': [2965, 2926, 2854, 1468, 1260, 653],
        'PC': [828, 1013, 1158, 1186, 1364, 1409, 1503, 1768, 2966],
        'CA': [600, 904, 1368, 1743],
        'PET': [720, 1094, 1241, 1713],
        '': []
    }
        
    # Filter data for the current polymer type
    list_ = mpsat_results.loc[(mpsat_results["true"] == true_type) & (mpsat_results["predicted"] == predicted_types), "File name"].to_list()
    selected = initial_db[initial_db["File name"].isin(list_)] 
    
    # Extract x-axis values from column names (assuming data starts from column index 7)
    x_values = np.array([float(col) for col in selected.columns[7:]])
    
    # Prepare y-axis values for each row
    y_values = selected.iloc[:, 7:].values
    file_names = selected["File name"].values  # Extract file names
    
    # Create the figure using Plotly
    fig = go.Figure()

    # Plot data with file name on hover
    for i in range(len(y_values)):
        fig.add_trace(go.Scatter(
            x=x_values, 
            y=y_values[i], 
            mode='lines', 
            line=dict(color='gray', width=1), 
            showlegend=False,
            name=file_names[i],  # Use file name in legend
            hovertemplate=f"File: {file_names[i]}<br>Wavenumber: %{{x}} cm⁻¹<br>Absorbance: %{{y}} a.u."
        ))

    # Add gray vertical lines at characteristic wavenumbers for both polymer types
    polymers_for_peaks = [true_type] + [item for item in predicted_types.split('+') if item not in [true_type]]
    colors = ['red', 'blue', 'green']
    for pol, col in zip(polymers_for_peaks, colors):
        for wn in wavenumbers[pol]:
            fig.add_vline(x=wn, line=dict(color=col, dash='dash', width=1.5), opacity=0.5)
    
    # Add legend entries
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', name=true_type+' spectra', line=dict(color='gray', width=2)))
    for pol, col in zip(polymers_for_peaks, colors):
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', name=pol+' lines', line=dict(color=col, width=2)))

    # Set layout properties
    fig.update_layout(
        title=f"{true_type} spectra classified as {predicted_types}",
        xaxis_title="Wavenumber, cm<sup>-1</sup>",
        yaxis_title="Absorbance, a.u.",
        xaxis=dict(range=[450, 4000]),
        yaxis=dict(range=[-0.02, 0.6]),
        showlegend=True,
        legend=dict(
            x=1.05, y=1, traceorder='normal', orientation='v', xanchor='left', yanchor='top'
        ),
        font=dict(size=12),
        height=500,
        margin=dict(l=50, r=150, t=50, b=50)
    )

    # Save the plot as an interactive HTML file
    fig.write_html(f"./{true_type}={predicted_types}_interactive.html")

    return selected

In [ ]:
#END REQUIRED FUNCTIONS

In [ ]:
#MAIN

In [ ]:
red_db_file_baseline  = "../../4_baseline_correction/MICROSCAN_database_baseline_corrected.csv"
red_db_baseline = pd.read_csv(red_db_file_baseline)
red_db_baseline.head()

In [ ]:
red_db_file  = "../../2_compiling_unified_database/MICROSCAN_database.csv"
red_db = pd.read_csv(red_db_file)
red_db.head()

In [ ]:
# Load CNN1D predictions
df = pd.read_csv('../../3_applying_CNN1D/manual_and_CNN1D_classification.csv')
df.head()

In [ ]:
#DRAW

In [ ]:
# Load CNN1D results
with open('../../3_applying_CNN1D/cnn1d_results_full_db.pickle', 'rb') as file:
    results_red = pickle.load(file)

In [ ]:
def evaluate(true_, predicted_, panel_label_symbol_):
    filenames   = df.loc[(df["true"] == true_) & (df["predicted"] == predicted_), "File name"].to_list()
    for i in filenames:
        print(i, results_red[i][0:3])

    dff = plot_data_single_panel_interactive(mpsat_results=df, initial_db=red_db_baseline, true_type=true_, predicted_types=predicted_, panel_label_symbol=panel_label_symbol_)

In [ ]:
for true_class in ['PE', 'PP', 'PS']:
    unique_predicted_values = df.loc[df["true"] == true_class, "predicted"].unique()
    for pred_class in tqdm(unique_predicted_values):
        evaluate(true_=true_class, predicted_=pred_class, panel_label_symbol_='')

In [ ]:
!mkdir -p ./html_files
!mv ./*.html ./html_files/